# 02 — Activation Analysis

**Amaç:** Forward hooks kullanarak internal activations gözlemlemek, class-wise feature davranışını incelemek ve seçilmiş örneklerde internal activation karşılaştırması yapmak. Observation tek başına nedensellik kanıtı değildir.

In [ ]:
# Temel kütüphaneleri içe aktarırız; veri yükleme, model çalıştırma ve grafik için kullanılır.
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path: sys.path.append(ROOT)
from src.model import build_model
from src.hooks import register_activation_hooks, remove_hooks
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = build_model(42).to(device)
model.load_state_dict(torch.load('../results/baseline_model.pt', map_location=device))
model.eval()
transform = transforms.ToTensor()
test = datasets.MNIST('data', train=False, download=True, transform=transform)
loader = DataLoader(test, batch_size=256, shuffle=False)
print('device:', device, 'test samples:', len(test))

In [ ]:
# ReLU2 activationlarını test dataset'inin tamamı için toplarız; bu matrix feature analizi için kullanılır.
activations, handles = register_activation_hooks(model, ['net.2', 'net.4'])
all_acts = {'net.2': [], 'net.4': []}
all_labels = []
with torch.no_grad():
    for x, y in loader:
        _ = model(x.to(device))
        for name in all_acts: all_acts[name].append(activations[name].cpu())
        all_labels.append(y)
for name in all_acts:
    A = torch.cat(all_acts[name], dim=0)
    Y = torch.cat(all_labels, dim=0)
    print(name, 'shape=', tuple(A.shape), 'mean=', float(A.mean()), 'max=', float(A.max()), 'zero_ratio=', float((A == 0).float().mean()))
    class_means = torch.stack([A[Y == c].mean(dim=0) for c in range(10)])
    candidate = int(class_means.var(dim=0).argmax())
    print('candidate_neuron_by_class_variance:', candidate)
remove_hooks(handles)
A = torch.cat(all_acts['net.4'], dim=0)
Y = torch.cat(all_labels, dim=0)

In [ ]:
# Seçilen örneklerin internal activationlarını karşılaştırırız; örnek bazlı Glass Box gözlemi sağlar.
candidate_neurons = [47, 17, 57, 53, 28]
sample_indices = []
for cls in range(3):
    idx = int((Y == cls).nonzero(as_tuple=False)[0])
    sample_indices.append(idx)
print('selected sample indices:', sample_indices)
print('labels:', [int(Y[i]) for i in sample_indices])
sample_matrix = A[sample_indices][:, candidate_neurons].numpy()
plt.figure(figsize=(8, 4))
plt.imshow(sample_matrix, aspect='auto')
plt.xticks(range(len(candidate_neurons)), [f'N{n}' for n in candidate_neurons])
plt.yticks(range(len(sample_indices)), [f'Example {i} (Class {int(Y[idx])})' for i, idx in enumerate(sample_indices)])
plt.xlabel('Candidate neuron')
plt.ylabel('Selected example')
plt.title('Selected Examples: Candidate Internal Activations')
plt.colorbar(label='Activation')
plt.tight_layout()
plt.show()

## Çıktılar
- Activation matrix
- Ortalama aktivasyon
- Maksimum aktivasyon
- Zero ratio
- Sınıf bazlı activation davranışı
- Aday feature/neuron
- Seçilmiş örneklerde internal activation karşılaştırması

Aday seçim sonucu yalnızca hipotezdir. Nedensel rol ablation/intervention ile test edilmelidir.